In [41]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

True
Tesla T4
15.637086208 GB


In [42]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/sql-lora-project"
os.makedirs(PROJECT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
!pip install peft bitsandbytes accelerate trl datasets -q

In [44]:
from datasets import load_dataset

ds = load_dataset("b-mc2/sql-create-context")
print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 78577
    })
})
{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}


In [45]:
def format_example(ex):
    return {
        "text": f"### Schema:\n{ex['context']}\n\n### Question:\n{ex['question']}\n\n### SQL:\n{ex['answer']}"
    }

ds = ds["train"].map(format_example)
ds = ds.train_test_split(test_size=0.02, seed=42)
train_ds = ds["train"]
eval_ds = ds["test"]
print(len(train_ds), len(eval_ds))
print(train_ds[0]["text"])

77005 1572
### Schema:
CREATE TABLE Ref_colors (color_description VARCHAR, color_code VARCHAR); CREATE TABLE products (product_name VARCHAR, product_description VARCHAR, color_code VARCHAR)

### Question:
List the names, color descriptions and product descriptions of products with category "Herbs".

### SQL:
SELECT T1.product_name, T2.color_description, T1.product_description FROM products AS T1 JOIN Ref_colors AS T2 ON T1.color_code = T2.color_code WHERE product_category_code = "Herbs"


In [46]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-0.5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [47]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [50]:
from trl import SFTTrainer, SFTConfig

config = SFTConfig(
    output_dir=f"{PROJECT_DIR}/checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    dataset_text_field="text",
    max_length=512,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    args=config,
)

In [ ]:
trainer.train()

In [52]:
adapter_path = f"{PROJECT_DIR}/sql-lora-adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print("Saved to", adapter_path)

Saved to /content/drive/MyDrive/sql-lora-project/sql-lora-adapter


In [53]:
def generate_sql(schema, question, max_new_tokens=100):
    prompt = f"### Schema:\n{schema}\n\n### Question:\n{question}\n\n### SQL:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result[len(prompt):].strip()

sample = eval_ds[0]
print("Question:", sample["question"])
print("Reference SQL:", sample["answer"])
print("Generated SQL:", generate_sql(sample["context"], sample["question"]))

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


Question: When Essendon played away; where did they play?
Reference SQL: SELECT venue FROM table_name_50 WHERE away_team = "essendon"
Generated SQL: SELECTulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulanulan


In [58]:
model.eval()
model.config.use_cache = True
model.gradient_checkpointing_disable()

def generate_sql(schema, question, max_new_tokens=100):
    prompt = f"### Schema:\n{schema}\n\n### Question:\n{question}\n\n### SQL:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result[len(prompt):].strip()

sample = eval_ds[0]
print("Question:", sample["question"])
print("Reference SQL:", sample["answer"])
print("Generated SQL:", generate_sql(sample["context"], sample["question"]))

Question: When Essendon played away; where did they play?
Reference SQL: SELECT venue FROM table_name_50 WHERE away_team = "essendon"
Generated SQL: SELECT venue FROM table_name_50 WHERE away_team = "essendon"


In [59]:
import re

def normalize(sql):
    return re.sub(r"\s+", " ", sql.strip().lower()).rstrip(";")

n_eval = 200
correct = 0
examples_log = []

for i in range(n_eval):
    ex = eval_ds[i]
    pred = generate_sql(ex["context"], ex["question"])
    match = normalize(pred) == normalize(ex["answer"])
    correct += match
    examples_log.append((ex["question"], ex["answer"], pred, match))

print(f"Exact-match accuracy: {correct}/{n_eval} = {correct/n_eval:.2%}")

Exact-match accuracy: 42/200 = 21.00%


In [60]:
for q, ref, pred, match in examples_log[:10]:
    print("Q:", q)
    print("REF :", ref)
    print("PRED:", pred)
    print("MATCH:", match)
    print("-" * 50)

Q: When Essendon played away; where did they play?
REF : SELECT venue FROM table_name_50 WHERE away_team = "essendon"
PRED: SELECT venue FROM table_name_50 WHERE away_team = "essendon"
MATCH: True
--------------------------------------------------
Q: What is the lowest numbered game against Phoenix with a record of 29-17?
REF : SELECT MIN(game) FROM table_name_61 WHERE opponent = "phoenix" AND record = "29-17"
PRED: SELECT MIN(game) FROM table_name_61 WHERE opponent = "phoenix" AND record > 29
MATCH: False
--------------------------------------------------
Q: Who did the Texan's play on Week 4?
REF : SELECT opponent FROM table_name_37 WHERE week = "4"
PRED: SELECT opponent FROM table_name_37 WHERE week = "week 4"
MATCH: False
--------------------------------------------------
Q: Which Points have Touchdowns larger than 0, and an Extra points smaller than 0?
REF : SELECT SUM(points) FROM table_name_70 WHERE touchdowns > 0 AND extra_points < 0
PRED: SELECT SUM(points) FROM table_name_70 